# Fabry–Pérot interferometer: a compact forward and inverse model

This notebook introduces thermal line broadening, Doppler shift, etalon transmission, synthetic detector counts, and a simple line-of-sight wind fit. The physical and instrument functions are provided by the local `sbgea_keynote` package.

In [ ]:
from pathlib import Path
import sys
ROOT=next((p for p in (Path.cwd(),*Path.cwd().parents) if (p/"pyproject.toml").exists()),None)
if ROOT is None: raise FileNotFoundError("Abra o Jupyter dentro do checkout do projeto.")
sys.path.insert(0,str(ROOT/"src")) if str(ROOT/"src") not in sys.path else None
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display,Markdown
from sbgea_keynote.config import FPI
from sbgea_keynote.instruments.fabry_perot import (thermal_sigma_wavelength_m,wavelength_grid,emission_profile,
    detector_geometry,expected_interferogram,simulate_interferogram,
    retrieve_line_of_sight_wind,compare_instrument_precision)
NAVY,BLUE,GREEN,ORANGE="#082F73","#2686D0","#70AE54","#EC8135"
plt.rcParams.update({"figure.facecolor":"white","axes.facecolor":"white","font.size":12})
print("λ₀ = {:.4f} nm".format(FPI.line_center_m*1e9))
print("σ térmico = {:.3f} pm".format(thermal_sigma_wavelength_m()*1e12))
print("T = {:.1f} K · finesse = {:.1f}".format(FPI.gas_temperature_k,FPI.finesse))

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,4.8))
for v,color in zip([-120.,0.,120.],[BLUE,NAVY,ORANGE]):
    lam=wavelength_grid(FPI,wind_los_m_s=v)
    axs[0].plot((lam-FPI.line_center_m)*1e12,emission_profile(lam,v,FPI),
                color=color,lw=2,label="{:+.0f} m/s".format(v))
axs[0].set(xlabel="deslocamento em relação a λ₀ (pm)",
           ylabel="densidade espectral normalizada",title="Linha térmica e efeito Doppler")
axs[0].legend(frameon=False);axs[0].grid(alpha=.2)
xx,yy,radius,theta=detector_geometry(FPI)
im=axs[1].imshow(expected_interferogram(120.,FPI),origin="lower",cmap="viridis",
    extent=[xx.min()*1e3,xx.max()*1e3,yy.min()*1e3,yy.max()*1e3])
axs[1].set(xlabel="detector x (mm)",ylabel="detector y (mm)",title="Interferograma esperado · +120 m/s")
plt.colorbar(im,ax=axs[1],label="contagens esperadas")
plt.tight_layout();plt.show()

In [ ]:
truth=120.0
obs=simulate_interferogram(truth,seed=2026)
fit=retrieve_line_of_sight_wind(obs)
fig,ax=plt.subplots(figsize=(6.5,5.5))
im=ax.imshow(obs["counts"],origin="lower",cmap="viridis")
ax.set(title="Interferograma com ruído",xlabel="pixel x",ylabel="pixel y")
plt.colorbar(im,ax=ax,label="contagens");plt.show()
print("Vento prescrito (validação sintética): {:.1f} m/s".format(truth))
print("Vento na linha de visada recuperado: {:.2f} ± {:.2f} m/s".format(fit["wind_los_m_s"],fit["wind_std_m_s"]))
print("χ² reduzido: {:.2f}".format(fit["reduced_chi_square"]))
print("Convergiu:",fit["success"])

In [ ]:
rows=compare_instrument_precision(wind_los_m_s=120.,config=FPI,
    photon_scales=(400.,1600.,6400.),finesses=(12.,25.,40.))
photons=sorted(set(r["photons"] for r in rows))
finesses=sorted(set(r["finesse"] for r in rows))
fig,ax=plt.subplots(figsize=(8,5))
for finesse in finesses:
    vals=[next(r["wind_sigma_m_s"] for r in rows
               if r["photons"]==p and r["finesse"]==finesse) for p in photons]
    ax.plot(photons,vals,"o-",lw=2,label="finesse = {}".format(finesse))
ax.set_xscale("log")
ax.set(xlabel="escala de fótons (contagens)",ylabel="σ local do vento (m/s)",
       title="Precisão prevista pelo modelo idealizado")
ax.grid(alpha=.25);ax.legend(frameon=False);plt.show()
display(Markdown("O instrumento transforma o sinal Doppler em franjas mensuráveis. A precisão depende da geometria, finesse, fótons, fundo e ruído de leitura; erros de calibração e de modelo precisam ser avaliados separadamente."))

## Interpretation

The synthetic fit demonstrates parameter recovery under the assumed forward model. It does not include calibration drift, etalon defects, real sky background, or model discrepancy; these can dominate an operational instrument's error budget.